In [1]:
import pandas as pd
from BECancerResistome.processing.process import *
import os


In [20]:
def process_df(file,study,sheet_name=None,comparison=None):
    valid_studies = {'MC', 'GR', 'EG', 'RH'}
    valid_comparisons = {'RH_BRCA1_processed','RH_BRCA2_processed','RH_MCL1_processed',
        'RH_BCL2L1_processed','RH_PARP1_processed','MC_processed','GR_A549_ABE_CBE_MELJUSO_CBE_Pro',
        'GR_MELJUSO_ABE_Processed'}

    assert study in valid_studies, f"Invalid study: {study}. Must be one of {valid_studies}"
    if comparison is not None:
        assert comparison in valid_comparisons, f"Invalid comparison: {comparison}. Must be one of {valid_comparisons}"

    
    if study=='MC':
        df=pd.read_csv(f'data/3_Counts/MC/{file}')
        df.sort_values(by='guide')
        df=process_MC(df)
        df.to_csv("data/3_Counts/MC/MC_processed.csv", index=False)
        lfc_data,zscores=LFC_Z(df,comparison)
        lfc_data.to_csv("data/5_LFC/MC/MC_LFC_pDNA.csv", index=False)
        zscores.to_csv("data/4_Screen_zscores/MC/MC_zscores_pDNA.csv", index=False)
        
    if study=='GR':
        name=file.replace(r'.csv', '').replace(r'.tsv', '').replace(r'.txt', '')
        df=pd.read_csv(f"data/3_Counts/GR/{file}")
        df.sort_values(by="Construct Barcode") #all unique guides
        df=process_GR(df)
        df.to_csv(f"data/3_Counts/GR/GR_{name}_processed.csv", index=False)
        if comparison== 'GR_A549_ABE_CBE_MELJUSO_CBE_Pro':
            lfc_data, zscores=LFC_Z(df,"GR_A549_ABE_CBE_MELJUSO_CBE_Pro")
            lfc_data.to_csv("data/5_lfc/GR/GR_A549_ABE_CBE_MELJUSO_CBE_LFC.csv", index=False)
            zscores.to_csv("data/4_Screen_zscores/GR/GR_A549_ABE_CBE_MELJUSO_CBE_zscores.csv", index=False)
        if comparison=='GR_MELJUSO_ABE_Processed': 
            lfc_data, zscores=LFC_Z(df,"GR_MELJUSO_ABE_Processed")
            lfc_data.to_csv("data/5_lfc/GR/GR_MELJUSO_ABE_LFC.csv", index=False)
            zscores.to_csv("data/4_Screen_zscores/GR/GR_MELJUSO_ABE_zscores.csv", index=False)

    if study=='RH':
        if sheet_name=='BE3_HAP1':
            subset='PARP1_set'
            df=pd.read_excel(f"data/3_Counts/RH/{file}",header=None, sheet_name=sheet_name)
            df=process_RH(df,subset=subset)
            df.to_csv("data/3_Counts/RH/RH_PARP1_processed.csv", index=False)
            lfc_data,zscores=LFC_Z(df,'RH_PARP1_processed')
            lfc_data.to_csv("data/5_LFC/RH/RH_PARP1_LFC.csv", index=False)
            zscores.to_csv("data/4_Screen_zscores/RH/RH_PARP1_zscores.csv", index=False)

        if sheet_name=='BRCA1' or sheet_name=='BRCA2':
            subset='BRCA1_BRCA2_set'
            df=pd.read_excel(f"data/3_Counts/RH/{file}",header=None, sheet_name=sheet_name)
            df=process_RH(df,subset=subset)
            df.to_csv(f"data/3_Counts/RH/RH_{sheet_name}_processed.csv", index=False)
            if sheet_name=='BRCA1':
                lfc_data, zscores=LFC_Z(df,"RH_BRCA1_processed")
                lfc_data.to_csv("data/5_LFC/RH/RH_BRCA1_LFC.csv", index=False)
                zscores.to_csv("data/4_Screen_zscores/RH/RH_BRCA1_zscores.csv", index=False)
            if sheet_name=='BRCA2':
                lfc_data, zscores=LFC_Z(df,"RH_BRCA2_processed")
                lfc_data.to_csv("data/5_LFC/RH/RH_BRCA2_LFC.csv", index=False)
                zscores.to_csv("data/4_Screen_zscores/RH/RH_BRCA2_zscores.csv", index=False)
                                
        if sheet_name=='MCL1' or sheet_name=='BCL2L1':
            subset='BCL2L1_MCL1_set'
            df=pd.read_excel(f"data/3_Counts/RH/{file}",header=None, sheet_name=sheet_name)
            df=process_RH(df,subset=subset)
            df.to_csv(f"data/3_Counts/RH/RH_{sheet_name}_processed.csv", index=False)
            lfc_data, zscores=LFC_Z(df,"RH_MCL1_processed")
            lfc_data.to_csv("data/5_LFC/RH/RH_MCL1_LFC.csv", index=False)
            zscores.to_csv("data/4_Screen_zscores/RH/RH_MCL1_zscores.csv", index=False)
    return df,lfc_data,zscores
    


# MC

In [ ]:
df_MC,lfc_data_MC,zscores_MC=process_df('MC_readcouns.csv','MC',comparison='MC_processed_p')

# RH

In [ ]:
df_PARP,lfc_data_PARP,zscores_PARP=process_df('counts_PARP1.xlsx','RH',sheet_name='BE3_HAP1',comparison='RH_PARP1_processed')
df_PARP

In [ ]:
df_BRCA1,lfc_data_BRCA1,zscores_BRCA1=process_df('counts_BRCA1_BRCA2.xlsx','RH',sheet_name='BRCA1',comparison="RH_BRCA1_processed") #only CBE editors and Nan, no ABE

In [ ]:
df_BRCA2,lfc_data_BRCA2,zscores_BRCA2=process_df('counts_BRCA1_BRCA2.xlsx','RH',sheet_name='BRCA2',comparison="RH_BRCA2_processed") #only CBE editors and Nan, no ABE

In [ ]:
df_MCL1,lfc_data_MCL1,zscores_MCL1=process_df('counts_MCL1_BCL2L1.xlsx','RH',sheet_name='MCL1',comparison="RH_MCL1_processed")


In [ ]:
df_BCL2L1,lfc_data_BCL2L1,zscores_BCL2L1=process_df('counts_MCL1_BCL2L1.xlsx','RH',sheet_name='BCL2L1',comparison="RH_BCL2L1_processed")

# GR

In [ ]:
import pandas as pd

df_A549,lfc_data_A549,zscores_A549=process_df('counts-A549_ABE_CBE_MELJUSO_CBE.txt','GR',comparison="GR_A549_ABE_CBE_MELJUSO_CBE_Pro")

In [ ]:
df_MELJUSO,lfc_data_MELJUSO,zscores_MELJUSO=process_df('counts-MELJUSO_ABE.txt','GR',comparison="GR_MELJUSO_ABE_Processed")
